In [1]:
import torch
import numpy as np
from collections import Counter

from ipynb.fs.defs.trading_env import TradingEnv

In [2]:
def evaluate_agent(agent, feature_df):

    env = TradingEnv(feature_df)

    state = env.reset()
    done = False

    portfolio_values = []
    actions_taken = []

    while not done:

        action = agent.choose_action(state)

        next_state, reward, done = env.step(action)

        portfolio_values.append(env.portfolio_value)
        actions_taken.append(action)

        state = next_state

    final_value = env.portfolio_value

    total_return = (final_value - env.initial_cash) / env.initial_cash

    buy_hold_return = (
        env.prices.iloc[-1] - env.prices.iloc[0]
    ) / env.prices.iloc[0]

    profits = env.trade_profits

    wins = [p for p in profits if p > 0]
    losses = [p for p in profits if p <= 0]

    win_rate = len(wins) / len(profits) if profits else 0

    avg_win = sum(wins) / len(wins) if wins else 0
    avg_loss = sum(losses) / len(losses) if losses else 0

    profit_factor = (
        sum(wins) / abs(sum(losses))
        if losses else float("inf")
    )

    portfolio_values = np.array(portfolio_values)

    daily_returns = (
        portfolio_values[1:] - portfolio_values[:-1]
    ) / portfolio_values[:-1]

    sharpe = (
        np.mean(daily_returns)
        / np.std(daily_returns)
        * np.sqrt(252)
        if np.std(daily_returns) > 0 else 0
    )

    running_max = np.maximum.accumulate(portfolio_values)

    drawdowns = (
        portfolio_values - running_max
    ) / running_max

    max_drawdown = drawdowns.min()

    print(f"Final Portfolio Value: ${final_value:.2f}")
    print(f"Strategy Return: {total_return*100:.2f}%")
    print(f"Buy & Hold Return: {buy_hold_return*100:.2f}%")
    print(f"Win Rate: {win_rate:.2%}")
    print(f"Average Win: {avg_win:.4f}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Profit Factor: {profit_factor:.4f}")
    print(f"Sharpe Ratio: {sharpe:.4f}")
    print(f"Max Drawdown: {max_drawdown*100:.2f}%")

    return {
        "final_value": final_value,
        "strategy_return": total_return,
        "buy_hold_return": buy_hold_return,
        "win_rate": win_rate,
        "profit_factor": profit_factor,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "portfolio_values": portfolio_values,
        "actions": actions_taken
    }

In [ ]:
results_table = []

for ticker in test_stocks:

    prices = monte_carlo_data(
        ticker,
        "2024-01-01",
        "2025-01-01"
    ).squeeze()

    features = build_features(prices)

    res = evaluate_agent(agent,features)

    results_table.append({
        "ticker": ticker,
        "strategy": res["strategy_return"],
        "buyhold": res["buy_hold_return"],
        "sharpe": res["sharpe"],
        "drawdown": res["max_drawdown"]
    })

for row in results_table:
    print(row)